#  Titanic Dataset: Complete Data Cleaning & Transformation Pipeline

##  Project Overview
This notebook processes and prepares the **Titanic** passenger dataset using Data Cleaning, Structural Transformation, and Feature Engineering techniques to get the data ready for Exploratory Data Analysis (EDA) and Machine Learning models.

---

##  Summary of Pipeline Steps

### 1. Data Cleaning & Missing Values Handling
* **`Age`**: Imputed missing values using the **Median** to prevent skewness from outliers.
* **`Embarked`**: Imputed missing values using the **Mode** (most frequent value).
* **`Cabin`**: Replaced missing values with an **`"Unknown"`** category label.

### 2. Data Type Conversions
* **`Fare`**: Casted data type from `float` to `Integer` after processing to remove unnecessary decimals.

### 3. Feature Engineering
* **`Title`**: Extracted passenger titles (`Mr`, `Mrs`, `Miss`, etc.) from the `Name` column using Regular Expressions (`Regex`).
* **`FamilySize`**: Calculated total family members aboard by summing (`SibSp + Parch + 1`).
* **`Fare_Category`**: Binned ticket fares into three categories (`Low`, `Medium`, `High`) based on price ranges.

### 4. Advanced Aggregation & Joins
* **Aggregation**: Calculated `max`, `min`, and `mean`/`avg` fare prices grouped by port of embarkation (`Embarked`).
* **Merging / Joins**: Joined external description tables with the main dataset to map passenger classes (`Pclass`) to their detailed descriptions.

---

## Summary of Tech Stack & Methods Used

| Pipeline Step | Pandas Method | PySpark Equivalent |
| :--- | :--- | :--- |
| **Handling Missing Data** | `.fillna()` / `.median()` / `.mode()` | `df.fillna()` / `approxQuantile()` |
| **Casting Types** | `.astype(int)` | `col().cast("integer")` |
| **String Extraction** | `.str.extract()` | `regexp_extract()` |
| **Conditional Logic** | `pd.cut()` / `np.select()` | `when().otherwise()` |
| **Aggregation** | `.groupby().agg()` | `.groupBy().agg()` |
| **Data Merging** | `pd.merge()` | `.join()` |

---

In [5]:
import pandas as pd

import urllib.request

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
urllib.request.urlretrieve(url, "./titanic.csv")


('./titanic.csv', <http.client.HTTPMessage at 0x2447d36cfc0>)

In [7]:
df=pd.read_csv('titanic.csv')

In [8]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
pclass1_over30=df[(df['Age']>30)&(df['Pclass']==1)]
pclass1_over30

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
30,31,0,1,"Uruchurtu, Don. Manuel E",male,40.0,0,0,PC 17601,27.7208,NaN,C
...,...,...,...,...,...,...,...,...,...,...,...,...
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S
867,868,0,1,"Roebling, Mr. Washington Augustus II",male,31.0,0,0,PC 17590,50.4958,A24,S
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
872,873,0,1,"Carlsson, Mr. Frans Olof",male,33.0,0,0,695,5.0000,B51 B53 B55,S


In [12]:
survived=df[(df['Survived']==1)&(df['Embarked']=="C")]
survived

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
36,37,1,3,"Mamee, Mr. Hanna",male,NaN,0,0,2677,7.2292,NaN,C
...,...,...,...,...,...,...,...,...,...,...,...,...
866,867,1,2,"Duran y More, Miss. Asuncion",female,27.0,1,0,SC/PARIS 2149,13.8583,NaN,C
874,875,1,2,"Abelson, Mrs. Samuel (Hannah Wizosky)",female,28.0,1,0,P/PP 3381,24.0000,NaN,C
875,876,1,3,"Najib, Miss. Adele Kiamie ""Jane""",female,15.0,0,0,2667,7.2250,NaN,C
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C


In [ ]:
import numpy as np
df['Age_Group']=np.where(df['Age']<18 , 'Child','Adult')
df.head(10)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_Group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Adult
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Adult
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Adult
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,Adult
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Adult
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,Child
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,Adult
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,Child


In [ ]:
# df['avg_group']=df['Avg'].apply(lambda age:'child' if age<18 also 'Adult')

In [20]:
df['Age']=df['Age'].fillna(df['Age'].mean())
print(df[['Name', 'Age']].head(10))

                                                Name        Age
0                            Braund, Mr. Owen Harris  22.000000
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.000000
2                             Heikkinen, Miss. Laina  26.000000
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.000000
4                           Allen, Mr. William Henry  35.000000
5                                   Moran, Mr. James  29.699118
6                            McCarthy, Mr. Timothy J  54.000000
7                     Palsson, Master. Gosta Leonard   2.000000
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  27.000000
9                Nasser, Mrs. Nicholas (Adele Achem)  14.000000


In [22]:
top_5_fare=df.sort_values(by="Fare" , ascending = False)
top_5_fare.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_Group
258,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C,Adult
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C,Adult
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C,Adult
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S,Adult
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S,Adult


In [26]:
df_filter=df[(df['Embarked']=="C") & (df['Fare']>100)]
df_filter[["Name", "Pclass", "Fare", "Embarked"]]

,Name,Pclass,Fare,Embarked
31,"Spencer, Mrs. William Augustus (Marie Eugenie)",1,146.5208,C
118,"Baxter, Mr. Quigg Edmond",1,247.5208,C
195,"Lurette, Miss. Elise",1,146.5208,C
215,"Newell, Miss. Madeleine",1,113.2750,C
258,"Ward, Miss. Anna",1,512.3292,C
299,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",1,247.5208,C
306,"Fleming, Miss. Margaret",1,110.8833,C
307,"Penasco y Castellana, Mrs. Victor de Satode (M...",1,108.9000,C
311,"Ryerson, Miss. Emily Borie",1,262.3750,C
319,"Spedden, Mrs. Frederic Oakley (Margaretta Corn...",1,134.5000,C


In [27]:
ports_pd = pd.DataFrame({
    'Embarked_Code': ['C', 'Q', 'S'],
    'Port_Name': ['Cherbourg', 'Queenstown', 'Southampton']
})

In [30]:
df_pd_joined = pd.merge(
    df,
    ports_pd,
    left_on='Embarked',
    right_on='Embarked_Code',
    how='left'
)
df_pd_joined[["PassengerId","Name","Embarked","Port_Name"]]

,PassengerId,Name,Embarked,Port_Name
0,1,"Braund, Mr. Owen Harris",S,Southampton
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,Cherbourg
2,3,"Heikkinen, Miss. Laina",S,Southampton
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,Southampton
4,5,"Allen, Mr. William Henry",S,Southampton
...,...,...,...,...
886,887,"Montvila, Rev. Juozas",S,Southampton
887,888,"Graham, Miss. Margaret Edith",S,Southampton
888,889,"Johnston, Miss. Catherine Helen ""Carrie""",S,Southampton
889,890,"Behr, Mr. Karl Howell",C,Cherbourg


In [37]:
classes_data = ({
    'Class_ID':[1,2,3],
    'Class_Description':['First Class - Luxury','Second Class - Mid Range','Third Class - Economy']
})
classes_df = pd.DataFrame(classes_data)
classes_df=pd.merge(
    df,
    classes_df,
    left_on='Pclass',
    right_on='Class_ID',
    how='left'
    )
classes_df[['PassengerId', 'Name', 'Pclass', 'Class_Description']]

,PassengerId,Name,Pclass,Class_Description
0,1,"Braund, Mr. Owen Harris",3,Third Class - Economy
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,First Class - Luxury
2,3,"Heikkinen, Miss. Laina",3,Third Class - Economy
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,First Class - Luxury
4,5,"Allen, Mr. William Henry",3,Third Class - Economy
...,...,...,...,...
886,887,"Montvila, Rev. Juozas",2,Second Class - Mid Range
887,888,"Graham, Miss. Margaret Edith",1,First Class - Luxury
888,889,"Johnston, Miss. Catherine Helen ""Carrie""",3,Third Class - Economy
889,890,"Behr, Mr. Karl Howell",1,First Class - Luxury


In [38]:
df['Fare'] = df['Fare'].astype(int)
print(df[['PassengerId', 'Name', 'Fare']].head(5))

   PassengerId                                               Name  Fare
0            1                            Braund, Mr. Owen Harris     7
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...    71
2            3                             Heikkinen, Miss. Laina     7
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)    53
4            5                           Allen, Mr. William Henry     8


In [39]:
df['Cabin'] = df['Cabin'].fillna("Unknown")

In [42]:
df['name_length']=df['Name'].str.len()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_Group,name_length
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7,Unknown,S,Adult,23
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71,C85,C,Adult,51
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7,Unknown,S,Adult,22
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53,C123,S,Adult,44
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8,Unknown,S,Adult,24


In [43]:
df_grouped_fare=df.groupby('Embarked').agg(
    Max_Fare=('Fare', 'max'),
    Min_Fare=('Fare', 'min'),
    Avg_Fare=('Fare', 'mean')
).reset_index()
df_grouped_fare

,Embarked,Max_Fare,Min_Fare,Avg_Fare
0,C,512,4,59.523810
1,Q,90,6,12.675325
2,S,263,0,26.684783


In [45]:
bins = [-1, 15, 50, float('inf')]
labels = ['Low', 'Medium', 'High']
df['Fare_Category'] = pd.cut(df['Fare'], bins=bins, labels=labels)

In [46]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [47]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_Group,name_length,Fare_Category,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000000,1,0,A/5 21171,7,Unknown,S,Adult,23,Low,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000000,1,0,PC 17599,71,C85,C,Adult,51,High,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000000,0,0,STON/O2. 3101282,7,Unknown,S,Adult,22,Low,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1,0,113803,53,C123,S,Adult,44,High,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.000000,0,0,373450,8,Unknown,S,Adult,24,Low,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.000000,0,0,211536,13,Unknown,S,Adult,21,Low,Rev
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.000000,0,0,112053,30,B42,S,Adult,28,Medium,Miss
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,29.699118,1,2,W./C. 6607,23,Unknown,S,Adult,40,Medium,Miss
889,890,1,1,"Behr, Mr. Karl Howell",male,26.000000,0,0,111369,30,C148,C,Adult,21,Medium,Mr
